In [6]:
import sys
from pathlib import Path

# notebooks/ -> repo root
REPO_ROOT = Path.cwd().resolve()
print(REPO_ROOT)
sys.path.insert(0, str(REPO_ROOT))

/Users/nolan/Documents/GitHub/active-passive-alternations


In [12]:
from pathlib import Path
from conllu import parse_incr
from src.units.sentence import ActiveSentence, PassiveSentence

In [13]:
DATA_PATH = Path("data/en_gum-ud-train.conllu")
if not DATA_PATH.exists(): DATA_PATH = Path("../data/en_gum-ud-train.conllu")
DATA_PATH

PosixPath('data/en_gum-ud-train.conllu')

In [16]:
with DATA_PATH.open("r", encoding="utf-8") as f:
    sents = list(parse_incr(f))

In [17]:
len(sents)

10224

In [18]:
def sent_text(s):
    return s.metadata.get("text") or " ".join(t["form"] for t in s if isinstance(t.get("id"), int))

In [21]:
def as_active(sent):
    try: return ActiveSentence(sent)
    except ValueError: return None

In [22]:
active_rows = [(i, as_active(s)) for i, s in enumerate(sents)]
active_rows = [(i, a) for i, a in active_rows if a is not None]
len(active_rows)

2304

In [27]:
active_rows[0]

(8,
 [{'id': 1,
   'form': 'Do',
   'lemma': 'do',
   'upos': 'AUX',
   'xpos': 'VBP',
   'feats': {'Mood': 'Ind',
    'Number': 'Plur',
    'Person': '3',
    'Tense': 'Pres',
    'VerbForm': 'Fin'},
   'head': 4,
   'deprel': 'aux',
   'deps': [('aux', 4)],
   'misc': {'Discourse': 'joint-list_m:9->7:0:sem-lxchn-50-51,75-76-_+syn-prl-56,65-66,78-gold',
    'PDTB': 'Implicit:Expansion.Conjunction:and:_:56-65:66-78'},
   'inflection': 'VBP',
   'children': []},
  {'id': 2,
   'form': 'museum',
   'lemma': 'museum',
   'upos': 'NOUN',
   'xpos': 'NN',
   'feats': {'Number': 'Sing'},
   'head': 3,
   'deprel': 'compound',
   'deps': [('compound', 3)],
   'misc': {'Entity': '(23-abstract-new-nnssn-cf2-2-sgl(24-organization-new-nnsnn-cf5-1-coref)'},
   'inflection': 'NN',
   'children': []},
  {'id': 3,
   'form': 'labels',
   'lemma': 'label',
   'upos': 'NOUN',
   'xpos': 'NNS',
   'feats': {'Number': 'Plur'},
   'head': 4,
   'deprel': 'nsubj',
   'deps': [('nsubj', 4)],
   'misc': {'En

In [25]:
i, a = active_rows[0]
print(sents[i].metadata.get("sent_id"))
print("Active:", sent_text(sents[i]))
print("Passive:", a.passivize().text)

GUM_academic_art-9
Active: Do museum labels have an impact on how people look at artworks?
Passive: Do museum labels have an impact on how people look at artworks?


In [30]:
for i, a in active_rows[:5]:
    print(sents[i].metadata.get("sent_id"))
    print("Active:", sent_text(sents[i]))
    print("Passive:", a.passivize().text, "\n")

GUM_academic_art-9
Active: Do museum labels have an impact on how people look at artworks?
Passive: Do museum labels have an impact on how people look at artworks? 

GUM_academic_art-12
Active: This paper describes a collaborative pilot project focusing on a unique collection of 17th Century Zurbarán paintings.
Passive: A collaborative pilot project focusing on a unique collection of 17th Century Zurbarán paintings is described by this paper. 

GUM_academic_art-17
Active: We will discuss the potential implications of these techniques and our understanding of visual behaviours on museum and gallery practice.
Passive: The potential implications of these techniques and our understanding of visual behaviours on museum and gallery practice will be discussed by us. 

GUM_academic_art-18
Active: The project brings together established research strengths in Spanish art history, experimental psychology, digital humanities, and museum studies to explore, using eye-tracking techniques, aesthetic 

In [32]:
# to handle errors
def safe_passivize(a):
    try: return a.passivize().text
    except Exception as e: return f"ERROR: {e}"

In [33]:
rows = [(s.metadata.get("sent_id"), sent_text(s), a, safe_passivize(a) if a else None) for s in sents for a in [as_active(s)]]
len(rows)

10224

In [34]:
eligible = sum(a is not None for _, _, a, _ in rows)
converted = sum(a is not None and out != txt for _, txt, a, out in rows)
print({"eligible": eligible, "converted": converted, "unchanged": eligible - converted, "exceptions": sum(str(out).startswith("ERROR:") for _, _, a, out in rows if a is not None)})

{'eligible': 2304, 'converted': 1616, 'unchanged': 688, 'exceptions': 0}


In [36]:
changed = [(sid, txt, out) for sid, txt, a, out in rows if a is not None and out != txt][:12]
for sid, txt, out in changed: print(f"[{sid}]\nA: {txt}\nP: {out}\n")

[GUM_academic_art-12]
A: This paper describes a collaborative pilot project focusing on a unique collection of 17th Century Zurbarán paintings.
P: A collaborative pilot project focusing on a unique collection of 17th Century Zurbarán paintings is described by this paper.

[GUM_academic_art-17]
A: We will discuss the potential implications of these techniques and our understanding of visual behaviours on museum and gallery practice.
P: The potential implications of these techniques and our understanding of visual behaviours on museum and gallery practice will be discussed by us.

[GUM_academic_art-18]
A: The project brings together established research strengths in Spanish art history, experimental psychology, digital humanities, and museum studies to explore, using eye-tracking techniques, aesthetic reactions to digital representations of the individual Zurbarán artworks as well as the significance of the collection as a whole.
P: Established research strengths in Spanish art history, 

In [44]:
changed = [(sid, txt, out) for sid, txt, a, out in rows if a is not None and out != txt and not str(out).startswith("ERROR:")][:12]
for sid, txt, out in changed: print(f"[{sid}]\nA: {txt}\nP: {out}\n")

[GUM_academic_art-12]
A: This paper describes a collaborative pilot project focusing on a unique collection of 17th Century Zurbarán paintings.
P: A collaborative pilot project focusing on a unique collection of 17th Century Zurbarán paintings is described by this paper.

[GUM_academic_art-17]
A: We will discuss the potential implications of these techniques and our understanding of visual behaviours on museum and gallery practice.
P: The potential implications of these techniques and our understanding of visual behaviours on museum and gallery practice will be discussed by us.

[GUM_academic_art-18]
A: The project brings together established research strengths in Spanish art history, experimental psychology, digital humanities, and museum studies to explore, using eye-tracking techniques, aesthetic reactions to digital representations of the individual Zurbarán artworks as well as the significance of the collection as a whole.
P: Established research strengths in Spanish art history, 

In [45]:
unchanged = [(sid, txt, out) for sid, txt, a, out in rows if a is not None and out == txt]
for sid, txt, out in unchanged[:12]: print(f"[{sid}]\nA: {txt}\nP: {out}\n")
len(unchanged)

[GUM_academic_art-9]
A: Do museum labels have an impact on how people look at artworks?
P: Do museum labels have an impact on how people look at artworks?

[GUM_academic_art-26]
A: It has a long history in scholarship (Baron & Beresford 2014), but many key aspects of its production and significance have not yet been fully understood.
P: It has a long history in scholarship (Baron & Beresford 2014), but many key aspects of its production and significance have not yet been fully understood.

[GUM_academic_art-28]
A: This pilot project primarily investigated how participants visually explore artworks and provides new insights into the potential eye-tracking has to transform the ways we understand visual processing in arts and culture and at the same time offer a direct way of studying several important factors of a museum visit, namely to assess the effects of label characteristics on visitor visual behaviour.
P: This pilot project primarily investigated how participants visually explore 

688

In [46]:
neg = [(sid, txt, out) for sid, txt, a, out in rows if a is not None and " not " in f" {txt.lower()} "][:10]
for sid, txt, out in neg: print(f"[{sid}]\nA: {txt}\nP: {out}\n")

[GUM_academic_art-26]
A: It has a long history in scholarship (Baron & Beresford 2014), but many key aspects of its production and significance have not yet been fully understood.
P: It has a long history in scholarship (Baron & Beresford 2014), but many key aspects of its production and significance have not yet been fully understood.

[GUM_academic_economics-20]
A: These institutions benefit not only the elites, but society as a whole.
P: Not only the elites, but society as a whole are benefited by these institutions.

[GUM_academic_epistemic-4]
A: In fact, many, if not all, the markers of expertise identified by philosophers enjoy widespread recognition.
P: In fact, widespread recognition is enjoyed by many, if not all, the markers of expertise identified by philosophers.

[GUM_academic_huh-4]
A: But it is not that a word can have just any vocal sound.
P: But it is not that a word can have just any vocal sound.

[GUM_academic_huh-8]
A: From a systematic comparison of 10 spoken langu